# Information Extraction

In [7]:
import pandas as pd
import re
import emoji

import spacy
from spacy.cli import download
import textacy

from collections import Counter
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib.gridspec import GridSpec
from wordcloud import WordCloud

pd.set_option('display.max_colwidth', None)

Pre-Processing

In [3]:
df = pd.read_csv('../data/sentiment-emotion-labelled_Dell_tweets.csv')

# Cancello prima colonna
df.drop(columns=df.columns[0], axis = 1, inplace=True)
print(f"Dimensioni: {df.shape[0]} righe, {df.shape[1]} colonne\n")
print(f"\nColonne presenti:")
print(df.columns.tolist())
print(f"\nTipi di dato:")
print(df.dtypes)
print(f"\nPrime 5 righe:")
df.head()

Dimensioni: 24970 righe, 8 colonne


Colonne presenti:
['Datetime', 'Tweet Id', 'Text', 'Username', 'sentiment', 'sentiment_score', 'emotion', 'emotion_score']

Tipi di dato:
Datetime            object
Tweet Id             int64
Text                object
Username            object
sentiment           object
sentiment_score    float64
emotion             object
emotion_score      float64
dtype: object

Prime 5 righe:


,Datetime,Tweet Id,Text,Username,sentiment,sentiment_score,emotion,emotion_score
0,2022-09-30 23:29:15+00:00,1575991191170342912,"@Logitech @apple @Google @Microsoft @Dell @Lenovo #WhatIf QWERTY were modified for programmers so things like brackets, parens, quotes, operators, etc. moved to the middle near G-H splitting left/right brackets to separate hands, and relieving the less dextrous pinky finger?",ManjuSreedaran,neutral,0.853283,anticipation,0.587121
1,2022-09-30 21:46:35+00:00,1575965354425131008,@MK_habit_addict @official_stier @MortalKombat @newzealand She's getting a new @Dell #laptop when the one she has one only 2-3 years old. (More than triple the price - though much higher utility). https://t.co/7WvkCw7vQf,MiKeMcDnet,neutral,0.519470,joy,0.886913
2,2022-09-30 21:18:02+00:00,1575958171423752203,"As @CRN celebrates its 40th anniversary, Bob Faletra and @stevenjburke spoke with me about the milestones, companies and personalities that helped build the channel. https://t.co/stiuBObP1O #CRN40 #podcast #internationalpodcastday @Cisco @Microsoft @HPE @hp @Dell @intel",jfollett,positive,0.763791,joy,0.960347
3,2022-09-30 20:05:24+00:00,1575939891485032450,@dell your customer service is horrible especially agent syedfaisal who has made this experience of purchasing a new computer downright awful and I’ll reconsider ever buying a Dell in the future @DellTech,daveccarr,negative,0.954023,anger,0.983203
4,2022-09-30 20:03:17+00:00,1575939359160750080,@zacokalo @Dell @DellCares @Dell give the man what he paid for!,heycamella,neutral,0.529170,anger,0.776124


In [4]:
# Valori null
print("\n VALORI MANCANTI:")
print(df.isnull().sum())

if df.isnull().sum().sum() == 0:
    print("✓ Nessun valore mancante!")

# Duplicati
print("\n DUPLICATI:")
print(f"  • Righe duplicate: {df.duplicated().sum()}")
print(f"  • Tweet duplicati (stesso testo): {df['Text'].duplicated().sum()}")
print(f"  • Tweet ID duplicati: {df['Tweet Id'].duplicated().sum()}")

# Mostra esempi di duplicati
if df['Text'].duplicated().sum() > 0:
    print("\n Esempi di tweet duplicati:")
    dup_mask = df['Text'].duplicated(keep=False)
    print(df[dup_mask][['Text', 'sentiment', 'emotion']].head(6))

# I duplicati vengono rimossi per evitare overfitting
print("RIMOZIONE DUPLICATI")

# Rimozione duplicati basata sul testo
df_clean = df.drop_duplicates(subset='Text', keep='first')

# Statistiche dopo la rimozione
print(f"Dataset dopo rimozione duplicati: {len(df_clean)} righe")
print(f"Righe rimosse: {len(df) - len(df_clean)}")
print(f"Percentuale dati mantenuti: {len(df_clean)/len(df)*100:.2f}%")

print(f"\nVerifica: Tweet duplicati rimasti: {df_clean['Text'].duplicated().sum()}")

df = df_clean


 VALORI MANCANTI:
Datetime           0
Tweet Id           0
Text               0
Username           0
sentiment          0
sentiment_score    0
emotion            0
emotion_score      0
dtype: int64
✓ Nessun valore mancante!

 DUPLICATI:
  • Righe duplicate: 0
  • Tweet duplicati (stesso testo): 331
  • Tweet ID duplicati: 0

 Esempi di tweet duplicati:
                                                                                                                                      Text  \
32   @ashu_k7 @Dell Pathetic!!!!! I Dont mind taking legal action, this is deficency of service of which the customer is nt getting help..   
36   @ashu_k7 @Dell Pathetic!!!!! I Dont mind taking legal action, this is deficency of service of which the customer is nt getting help..   
68                                                                                                                 @Dell That’s very great   
154                                                                        

## Keyphrase Extraction

Demojizing, eliminazione di menzioni e link, filtro hashtag

In [5]:
# Dizionario per contrazioni e abbreviazioni
CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "ll": " will",
    "'ve": " have",
    "'m": " am",

    # Twitter / informal
    "ty": "thank you",
    "thx": "thank you",
    "pls": "please",
    "plz": "please",
    "u": "you",
    "ur": "your",
    "imo": "in my opinion",
    "idk": "i do not know",
    "btw": "by the way"
}


# Funzione per espandere le contrazioni
def expand_contractions(text, contractions=CONTRACTIONS):
    for contr, full in contractions.items():
        pattern = r'\b' + re.escape(contr) + r'\b'
        text = re.sub(pattern, full, text)
    return text


def clean_tweet(text):
    text = text.lower()                                     # Conversione di tutti i tweet in Lower-case
    text = re.sub(r'http\S+|https\S+|www\S+', ' ', text)    # Rimozione URL
    text = expand_contractions(text)                        # Espansione delle contrazioni (es: ty --> thank you)
    text = emoji.demojize(text)                             # Conversione emoji in testo descrittivo
    text = text.replace(":", " ")                           # Rimozione ":" negli emoji convertiti in testo (es: :smile: --> smile)
    text = re.sub(r'@\w+', ' ', text)                       # Rimozione menzioni, ovvero @
    text = re.sub(r'#(\w+)', r'\1', text)                   # Mantimento solo del testo degli hashtag
    text = re.sub(r'[•▪▫◦‣⁃]', ' ', text)                   # Rimozione bullet point e simboli simili
    text = re.sub(r"[^a-z0-9\s:._\-']", ' ', text)          # Rimozione caratteri NON testuali inutili
    text = re.sub(r'\s+', ' ', text).strip()                # Rimozione spazi multipli

    return text

df['clean_text'] = df['Text'].apply(clean_tweet)


KPE con SpaCy "en_core_web_sm" 

In [8]:
# Caricamento modello SpaCy
download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

# Funzione per ottenere le keyphrase con TextRank 
def extract_keyphrases(text, topn=5, aggregate=True):
    doc = textacy.make_spacy_doc(text, lang=nlp)
    
    # Otteniamo i termini chiave --> ottengo coppie termine-peso: [(term, weight), ...]
    keyterms = textacy.extract.keyterms.textrank(doc, normalize="lemma", topn=topn) 
    
    # Restituiamo solo le parole chiave
    keyphrases = [term for term, weight in keyterms]
    
    # Se richiesto, aggrega varianti simili
    if aggregate:
        keyphrases = list(textacy.extract.aggregate_term_variants(set(keyphrases)))
    
    return keyphrases, [chunk.text for chunk in textacy.extract.noun_chunks(doc)]

# Applicazione di TextRank a tutti i tweet
df['keyphrases_nounchunks'] = df['clean_text'].apply(lambda x: extract_keyphrases(x, topn=5, aggregate=True))

# Separarzione in due colonne per chiarezza
df['keyphrases_sm'] = df['keyphrases_nounchunks'].apply(lambda x: x[0])
df['noun_chunks_sm'] = df['keyphrases_nounchunks'].apply(lambda x: x[1])
df.drop(columns=['keyphrases_nounchunks'], inplace=True)

# Visualizzazione risultato completo
print(df[['clean_text', 'keyphrases_sm', 'noun_chunks_sm']].head())


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
                                                                                                                                                                                                                   clean_text  \
0  whatif qwerty were modified for programmers so things like brackets parens quotes operators etc. moved to the middle near g-h splitting left right brackets to separate hands and relieving the less dextrous pinky finger   
1                                                                                          she is getting a new laptop when the one she has one only 2-3 years old. more than triple the price - though much higher ut

KPE con SpaCy "en_core_web_trf"

In [ ]:
# Caricamento modello Transformer
download("en_core_web_trf")
nlp = spacy.load("en_core_web_trf")

def extract_keyphrases_trf(text, topn=5, min_len=2, aggregate=True):
    if not isinstance(text, str) or not text.strip():
        return [], []

    # Documento spaCy
    doc = textacy.make_spacy_doc(text, lang=nlp)

    noun_chunks = [
        chunk.text.strip()
        for chunk in textacy.extract.noun_chunks(doc)
        if len(chunk.text.split()) >= min_len
    ]

    # Se non ci sono candidati, fallback
    if not noun_chunks:
        return [], []

    # TextRank per ranking
    keyterms = textacy.extract.keyterms.textrank(
        doc,
        normalize="lemma",
        topn=topn * 3   
    )

    ranked_terms = [term for term, _ in keyterms]

    # Intersezione: vengono tenuti solo noun chunks rankati
    keyphrases = []
    for chunk in noun_chunks:
        for term in ranked_terms:
            if term in chunk or chunk in term:
                keyphrases.append(chunk)
                break

    keyphrases = list(dict.fromkeys(keyphrases))[:topn]

    if aggregate and keyphrases:
        keyphrases = list(textacy.extract.aggregate_term_variants(set(keyphrases)))

    return keyphrases, noun_chunks

# Applicazione al DataFrame
df["kp_chunks"] = df["clean_text"].apply(lambda x: extract_keyphrases_trf(x, topn=5, aggregate=True))

df["keyphrases_trf"] = df["kp_chunks"].apply(lambda x: x[0])
df["noun_chunks_trf"] = df["kp_chunks"].apply(lambda x: x[1])
df.drop(columns=["kp_chunks"], inplace=True)

print(df[["clean_text", "keyphrases_trf", "noun_chunks_trf"]].head())


KPE con SpaCy "en_core_web_trf" migliorato

In [ ]:
# Caricamento modello SpaCy Transformer
download("en_core_web_trf")
nlp2 = spacy.load("en_core_web_trf")

# Funzione di estrazione keyphrase per singolo tweet
def extract_keyphrases_trf_2(text, topn=5, min_len=2):
    if not isinstance(text, str) or not text.strip():
        return [], []

    # Documento spaCy
    doc = textacy.make_spacy_doc(text, lang=nlp2)

    noun_chunks = [
        chunk.text.strip()
        for chunk in textacy.extract.noun_chunks(doc)
        if len(chunk.text.split()) >= min_len
    ]

    if not noun_chunks:
        return [], []

    # TextRank
    keyterms = textacy.extract.keyterms.textrank(doc, normalize="lemma", topn=topn * 3)
    ranked_terms = [term for term, _ in keyterms]

    # Intersezione ranking e noun chunks
    keyphrases = []
    for chunk in noun_chunks:
        for term in ranked_terms:
            if term in chunk or chunk in term:
                keyphrases.append(chunk)
                break

    keyphrases = list(dict.fromkeys(keyphrases))[:topn]

    return keyphrases, noun_chunks

# Applicazione al DataFrame
df["kp_chunks"] = df["clean_text"].apply(lambda x: extract_keyphrases_trf_2(x, topn=5))

df["keyphrases_trf2"] = df["kp_chunks"].apply(lambda x: x[0])
df["noun_chunks_trf2"] = df["kp_chunks"].apply(lambda x: x[1])
df.drop(columns=["kp_chunks"], inplace=True)

In [ ]:
print(df[["clean_text", "noun_chunks_trf2", "keyphrases_trf2"]].head())

In [ ]:
# Flatten di tutte le keyphrase
all_keyphrases = [
    kp
    for kps in df["keyphrases_trf2"]
    for kp in kps
    if isinstance(kp, str) and kp.strip()
]

# Aggregazione varianti a livello corpus
aggregated_terms_list = textacy.extract.aggregate_term_variants(set(all_keyphrases))

variant_to_canonical = {}
for variants_set in aggregated_terms_list:
    canonical = sorted(variants_set)[0]  # scegliamo la forma "minore" come canonica
    for v in variants_set:
        variant_to_canonical[v] = canonical

# Normalizzazione
normalized_keyphrases = [
    variant_to_canonical.get(kp, kp)
    for kp in all_keyphrases
]

# Conteggio frequenze globali
kp_counter = Counter(normalized_keyphrases)

# DataFrame finale con ranking globale
kp_global_df = (
    pd.DataFrame(kp_counter.items(), columns=["keyphrase_trf2", "frequency"])
    .sort_values("frequency", ascending=False)
    .reset_index(drop=True)
)

print(" TOP KEY-PHRASES GLOBALI\n")
print(kp_global_df.head(20))

print("\n SAMPLE TWEET-LEVEL ")
print(df[["clean_text", "noun_chunks_trf2", "keyphrases_trf2"]].head())


### Plot

Keyphrase più frequenti

In [ ]:
# Frequenza delle keyphrases: conteggiare quanto spesso ogni keyphrase appare nei tweet.
# Plot implementati:
#   a) Bar plot delle top 10-20 keyphrases più frequenti.
#   b) Word cloud per avere una panoramica visiva rapida.

# Primo modello
all_keyphrases_sm = [
    list(kp)[0] if isinstance(kp, set) else kp
    for sublist in df['keyphrases_sm']
    for kp in sublist
]

counter_sm = Counter(all_keyphrases_sm)
top10_sm = counter_sm.most_common(10)
labels_sm, values_sm = zip(*top10_sm)


# Secondo modello
top10_trf = kp_global_df.head(10)

labels_trf = top10_trf["keyphrase_trf2"].values
values_trf = top10_trf["frequency"].values

freq_dict_trf = dict(zip(kp_global_df["keyphrase_trf2"], kp_global_df["frequency"]))


fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(18,12),
                         gridspec_kw={'width_ratios': [1, 2]})

axes[0, 0].barh(labels_sm, values_sm)
axes[0, 0].invert_yaxis()
axes[0, 0].set_title("Top 10 Keyphrases - en_core_web_sm")
axes[0, 0].set_xlabel("Frequenza")

wordcloud_sm = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate(" ".join(all_keyphrases_sm))

axes[0, 1].imshow(wordcloud_sm, interpolation='bilinear')
axes[0, 1].axis('off')
axes[0, 1].set_title("Wordcloud - en_core_web_sm")


axes[1, 0].barh(labels_trf, values_trf)
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Top 10 Keyphrases - en_core_web_trf")
axes[1, 0].set_xlabel("Frequenza")


wordcloud_trf = WordCloud(
    width=800,
    height=400,
    background_color='white'
).generate_from_frequencies(freq_dict_trf)

axes[1, 1].imshow(wordcloud_trf, interpolation='bilinear')
axes[1, 1].axis('off')
axes[1, 1].set_title("Wordcloud - en_core_web_trf")

plt.tight_layout()
plt.show()

Distribuzione della complessità informativa per tweet

In [ ]:
# Calcoliamo:
#    1) in media, quanti keyphrase produce un tweet di una certa lunghezza;
#    2) frequenza per numero di keyphrase: Quanti tweet hanno 1, 2, 3 keyphrase?

# Lunghezza tweet
df['tweet_length'] = df['clean_text'].apply(len)

# Numero keyphrase SM
df['num_keyphrases_sm'] = df['keyphrases_sm'].apply(len)

# Numero keyphrase TRF
df['num_keyphrases_trf'] = df['keyphrases_trf'].apply(len)

# Distribuzioni
kp_counts_sm = df['num_keyphrases_sm'].value_counts().sort_index()
kp_counts_trf = df['num_keyphrases_trf'].value_counts().sort_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Scatter SM
axes[0, 0].scatter(
    df['tweet_length'],
    df['num_keyphrases_sm'],
    alpha=0.6
)
axes[0, 0].set_xlabel("Lunghezza tweet")
axes[0, 0].set_ylabel("Numero keyphrase")
axes[0, 0].set_title("SM: Keyphrases vs Lunghezza")

# Distribuzione SM
axes[0, 1].bar(
    kp_counts_sm.index,
    kp_counts_sm.values
)
axes[0, 1].set_xlabel("Numero di keyphrase")
axes[0, 1].set_ylabel("Numero di tweet")
axes[0, 1].set_title("SM: Distribuzione keyphrase per tweet")

# Scatter TRF
axes[1, 0].scatter(
    df['tweet_length'],
    df['num_keyphrases_trf'],
    alpha=0.6
)
axes[1, 0].set_xlabel("Lunghezza tweet")
axes[1, 0].set_ylabel("Numero keyphrase")
axes[1, 0].set_title("TRF: Keyphrases vs Lunghezza")

# Distribuzione TRF
axes[1, 1].bar(
    kp_counts_trf.index,
    kp_counts_trf.values
)
axes[1, 1].set_xlabel("Numero di keyphrase")
axes[1, 1].set_ylabel("Numero di tweet")
axes[1, 1].set_title("TRF: Distribuzione keyphrase per tweet")

plt.tight_layout()
plt.show()

Overlap a livello di keyphrase (sm e trf)

In [ ]:
# Jaccard locale (per ogni tweet, confronto le keyphrase estratte da sm e trf e vedo quanto coincidono): Quanto i modelli concordano tweet per tweet
def normalize_kps(kp_list):
    normalized = []
    
    for kp in kp_list:
        if isinstance(kp, set):
            # Se è un set, prendiamo l'elemento interno
            normalized.append(list(kp)[0])
        else:
            normalized.append(kp)
    
    return normalized

def jaccard_similarity(list1, list2):
    list1 = normalize_kps(list1)
    list2 = normalize_kps(list2)
    
    set1 = set(list1)
    set2 = set(list2)
    
    if len(set1) == 0 and len(set2) == 0:
        return 1
    
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    
    return intersection / union if union != 0 else 0

# Calcolo Jaccard per ogni tweet
df['jaccard_sm_trf'] = df.apply(
    lambda row: jaccard_similarity(
        row['keyphrases_sm'],
        row['keyphrases_trf']
    ),
    axis=1
)

# Media globale
mean_jaccard = df['jaccard_sm_trf'].mean()

print("Jaccard medio:", mean_jaccard)
print("----------------------------------------------\n")

# Jaccard globale: quanto sm e trf condividono il vocabolario globale di keyphrase
all_sm = set()

for sublist in df['keyphrases_sm']:
    if isinstance(sublist, (list, set)):
        for kp in sublist:
            all_sm.add(str(kp))

all_trf = set()

for sublist in df['keyphrases_trf']:
    if isinstance(sublist, (list, set)):
        for kp in sublist:
            all_trf.add(str(kp))

intersection_global = len(all_sm & all_trf)
union_global = len(all_sm | all_trf)

jaccard_global = intersection_global / union_global

print("Keyphrase uniche SM:", len(all_sm))
print("Keyphrase uniche TRF:", len(all_trf))
print("Intersezione globale:", intersection_global)
print("Jaccard globale:", jaccard_global)

# Calcolo valori Venn
only_sm = len(all_sm - all_trf)
only_trf = len(all_trf - all_sm)
intersection = len(all_sm & all_trf)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Istogramma Jaccard
axes[0].hist(df['jaccard_sm_trf'], bins=20)
axes[0].axvline(
    mean_jaccard,
    linestyle='--',
    linewidth=2,
    label=f"Jaccard medio = {mean_jaccard:.3f}"
)
axes[0].set_xlabel("Jaccard similarity")
axes[0].set_ylabel("Numero di tweet")
axes[0].set_title("Distribuzione tweet-level keyphrase overlap: SM vs TRF")
axes[0].legend()

# Venn diagram
intersection_value=f"Jaccard globale = {jaccard_global:.3f}"
venn2(
    subsets=(only_sm, only_trf, intersection),
    set_labels=('spaCy sm', 'spaCy trf'),
    ax=axes[1]
)
axes[1].set_title("Global Keyphrase Overlap")

axes[1].text(
    0.5,                
    -0.1,              
    intersection_value,
    ha='center',
    fontsize=11,
    transform=axes[1].transAxes
)

plt.tight_layout()
plt.show()

# Named Entity Recognition (NER)

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

import spacy

import re
import emoji

pd.set_option('display.max_colwidth', None)

Pre-Processing

In [2]:
df = pd.read_csv('../data/sentiment-emotion-labelled_Dell_tweets.csv')

# Cancello prima colonna
df.drop(columns=df.columns[0], axis = 1, inplace=True)
print(f"Dimensioni: {df.shape[0]} righe, {df.shape[1]} colonne\n")
print(f"\nColonne presenti:")
print(df.columns.tolist())
print(f"\nTipi di dato:")
print(df.dtypes)
print(f"\nPrime 5 righe:")
df.head()

Dimensioni: 24970 righe, 8 colonne


Colonne presenti:
['Datetime', 'Tweet Id', 'Text', 'Username', 'sentiment', 'sentiment_score', 'emotion', 'emotion_score']

Tipi di dato:
Datetime            object
Tweet Id             int64
Text                object
Username            object
sentiment           object
sentiment_score    float64
emotion             object
emotion_score      float64
dtype: object

Prime 5 righe:


,Datetime,Tweet Id,Text,Username,sentiment,sentiment_score,emotion,emotion_score
0,2022-09-30 23:29:15+00:00,1575991191170342912,"@Logitech @apple @Google @Microsoft @Dell @Lenovo #WhatIf QWERTY were modified for programmers so things like brackets, parens, quotes, operators, etc. moved to the middle near G-H splitting left/right brackets to separate hands, and relieving the less dextrous pinky finger?",ManjuSreedaran,neutral,0.853283,anticipation,0.587121
1,2022-09-30 21:46:35+00:00,1575965354425131008,@MK_habit_addict @official_stier @MortalKombat @newzealand She's getting a new @Dell #laptop when the one she has one only 2-3 years old. (More than triple the price - though much higher utility). https://t.co/7WvkCw7vQf,MiKeMcDnet,neutral,0.519470,joy,0.886913
2,2022-09-30 21:18:02+00:00,1575958171423752203,"As @CRN celebrates its 40th anniversary, Bob Faletra and @stevenjburke spoke with me about the milestones, companies and personalities that helped build the channel. https://t.co/stiuBObP1O #CRN40 #podcast #internationalpodcastday @Cisco @Microsoft @HPE @hp @Dell @intel",jfollett,positive,0.763791,joy,0.960347
3,2022-09-30 20:05:24+00:00,1575939891485032450,@dell your customer service is horrible especially agent syedfaisal who has made this experience of purchasing a new computer downright awful and I’ll reconsider ever buying a Dell in the future @DellTech,daveccarr,negative,0.954023,anger,0.983203
4,2022-09-30 20:03:17+00:00,1575939359160750080,@zacokalo @Dell @DellCares @Dell give the man what he paid for!,heycamella,neutral,0.529170,anger,0.776124


In [3]:
# Valori null
print("\n VALORI MANCANTI:")
print(df.isnull().sum())

if df.isnull().sum().sum() == 0:
    print("✓ Nessun valore mancante!")

# Duplicati
print("\n DUPLICATI:")
print(f"  • Righe duplicate: {df.duplicated().sum()}")
print(f"  • Tweet duplicati (stesso testo): {df['Text'].duplicated().sum()}")
print(f"  • Tweet ID duplicati: {df['Tweet Id'].duplicated().sum()}")

# Mostra esempi di duplicati
if df['Text'].duplicated().sum() > 0:
    print("\n Esempi di tweet duplicati:")
    dup_mask = df['Text'].duplicated(keep=False)
    print(df[dup_mask][['Text', 'sentiment', 'emotion']].head(6))

# I duplicati vengono rimossi per evitare overfitting
print("RIMOZIONE DUPLICATI")

# Rimozione duplicati basata sul testo
df_clean = df.drop_duplicates(subset='Text', keep='first')

# Statistiche dopo la rimozione
print(f"Dataset dopo rimozione duplicati: {len(df_clean)} righe")
print(f"Righe rimosse: {len(df) - len(df_clean)}")
print(f"Percentuale dati mantenuti: {len(df_clean)/len(df)*100:.2f}%")

print(f"\nVerifica: Tweet duplicati rimasti: {df_clean['Text'].duplicated().sum()}")

df = df_clean


 VALORI MANCANTI:
Datetime           0
Tweet Id           0
Text               0
Username           0
sentiment          0
sentiment_score    0
emotion            0
emotion_score      0
dtype: int64
✓ Nessun valore mancante!

 DUPLICATI:
  • Righe duplicate: 0
  • Tweet duplicati (stesso testo): 331
  • Tweet ID duplicati: 0

 Esempi di tweet duplicati:
                                                                                                                                      Text  \
32   @ashu_k7 @Dell Pathetic!!!!! I Dont mind taking legal action, this is deficency of service of which the customer is nt getting help..   
36   @ashu_k7 @Dell Pathetic!!!!! I Dont mind taking legal action, this is deficency of service of which the customer is nt getting help..   
68                                                                                                                 @Dell That’s very great   
154                                                                        

## NER

### Demoji, rimozione url, ecc...

In [4]:
# Dizionario per contrazioni e abbreviazioni
CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "n't": " not",
    "'re": " are",
    "'s": " is",
    "'d": " would",
    "'ll": " will",
    "ll": " will",
    "'ve": " have",
    "'m": " am",

    # Twitter / informal
    "ty": "thank you",
    "thx": "thank you",
    "pls": "please",
    "plz": "please",
    "u": "you",
    "ur": "your",
    "imo": "in my opinion",
    "idk": "i do not know",
    "btw": "by the way"
}

# Funzione per espandere le contrazioni
def expand_contractions(text, contractions=CONTRACTIONS):
    for contr, full in contractions.items():
        pattern = r'\b' + re.escape(contr) + r'\b'
        text = re.sub(pattern, full, text)
    return text

def preprocess_tweet(text):
    text = re.sub(r'http\S+|https\S+|www\S+', '', text)    # Rimozione URL
    text = expand_contractions(text)                        # Espansione delle contrazioni (es: ty --> thank you)
    text = emoji.demojize(text)                             # Conversione emoji in testo descrittivo
    text = text.replace(":", " ")                           # Rimuozione ":" negli emoji convertiti in testo (es: :smile: --> smile)
    text = text.replace("_", " ")                           # Rimuozione "_" negli emoji convertiti in testo (es: smiling_face --> smiling face)
    text = re.sub(r'#(\w+)', r'\1', text)                   # Mantimento solo del testo degli hashtag: #esempio -> esempio
    text = re.sub(r'[•▪▫◦‣⁃]', ' ', text)                   # Rimozione bullet point e simboli simili
    text = re.sub(r"[^A-Za-z0-9@\s._\-']", " ", text)       # Rimozione caratteri testuali inutili (Mantiene @, punto, underscore e trattino)
    text = re.sub(r'\s+', ' ', text).strip()                # Rimozione spazi multipli
    # Menzioni (@): Si mantengono le menzioni (@username) così come sono
    return text

# Applicazione del preprocessing
df['clean_text'] = df['Text'].apply(preprocess_tweet)

### Analisi con modello SpaCY "en_core_web_sm" (English pipeline)

In [ ]:
# Carico il modello SpaCy
download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

# Named Entity Labels:
# CARDINAL, DATE, EVENT, FAC, GPE, LANGUAGE, LAW, LOC, MONEY, NORP, ORDINAL, ORG, PERCENT, PERSON, PRODUCT, QUANTITY, TIME, WORK_OF_ART

# Funzione per eseguire NER
def perform_ner(tweets):
    ner_results = []
    for tweet in tweets:
        doc = nlp(tweet)
        entities = [(ent.text, ent.label_) for ent in doc.ents]
        ner_results.append(entities)
    return ner_results

# Applico NER al dataset
df['entities_sm'] = perform_ner(df['clean_text'])

# Mostro il risultato
print(df[['Text', 'clean_text', 'entities_sm']].head(10))

### Analisi con modello SpaCy "en_core_web_trf" (English transformer)

In [ ]:
# Carico il modello SpaCy
download("en_core_web_trf")
nlp_trf = spacy.load("en_core_web_trf")

# NER: Named Entity Labels: 
# CARDINAL, DATE, EVENT, FAC, GPE, LANGUAGE, LAW, LOC, MONEY, NORP, ORDINAL, ORG, PERCENT, PERSON, PRODUCT, QUANTITY, TIME, WORK_OF_ART

# Funzione per eseguire NER
def perform_ner_trf(tweets):
    ner_results_trf = []
    for doc in nlp_trf.pipe(tweets, batch_size=32):
        entities = [(ent.text, ent.label_) for ent in doc.ents]
        ner_results_trf.append(entities)
    return ner_results_trf

# Applico NER al dataset
df['entities_trf'] = perform_ner_trf(df['clean_text'])

# Mostro il risultato
print(df[['Text', 'clean_text', 'entities_trf']].head(10))

### Plot

In [ ]:
# Estrazione delle label
all_labels = []
all_entities = []
for entities in df['entities_trf']:
    for ent_text, ent_label in entities:
        all_labels.append(ent_label)
        all_entities.append((ent_text, ent_label))

label_counts = Counter(all_labels)

label_df = pd.DataFrame(label_counts.items(), columns=['Label', 'Count'])
label_df = label_df.sort_values(by='Count', ascending=False)

top12_df = label_df.head(12)

# Numero entità per tweet
df['n_entities'] = df['entities_trf'].apply(len)
media_entita = 2.5471001258167947
media_display = round(media_entita, 2)

# Confronto ORG con le altre
total_entities = len(all_labels)
org_entities = label_counts.get("ORG", 0)
other_entities = total_entities - org_entities

perc_org = 51.81
perc_other = 48.19

# Top ORG entities
org_list = [ent_text for ent_text, ent_label in all_entities if ent_label == "ORG"]
org_counts = Counter(org_list)

top10_org = pd.DataFrame(org_counts.most_common(10), columns=['Organizzazione', 'Frequenza'])


fig, axs = plt.subplots(2, 2, figsize=(14, 10))


axs[0, 0].bar(top12_df['Label'], top12_df['Count'])
axs[0, 0].set_title("Top 12 etichette NER")
axs[0, 0].set_xlabel("NER Label")
axs[0, 0].set_ylabel("Frequenza")
axs[0, 0].tick_params(axis='x', rotation=45)


axs[0, 1].hist(df['n_entities'], bins=20)
axs[0, 1].axvline(media_entita, linestyle='dashed')
axs[0, 1].set_title("Numero di entità per tweet")
axs[0, 1].set_xlabel("Numero di entità")
axs[0, 1].set_ylabel("Frequenza")
axs[0, 1].text(
    0.95, 0.95,
    f"Numero medio di entità per tweet:\n{media_display}",
    transform=axs[0, 1].transAxes,
    ha='right',
    va='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)


axs[1, 0].pie(
    [org_entities, other_entities],
    labels=[f"ORG ({perc_org})", f"Altre ({perc_other})"],
    startangle=90
)
axs[1, 0].set_title("Distribuzione ORG vs Altre label")


axs[1, 1].bar(top10_org['Organizzazione'], top10_org['Frequenza'], color='skyblue')
axs[1, 1].set_title("Top 10 entità ORG più frequenti")
axs[1, 1].set_xlabel("Organizzazione")
axs[1, 1].set_ylabel("Frequenza")
axs[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

Plot post rimozione keyword "dell"

In [ ]:
# Si escludono le keywords con dell
exclude_keywords = {
    "@dell",
    "dell",
    "@dellcares",
    "delltech",
    "@delltech"
}

all_labels = []
all_entities = []

filtered_entities_column = []

for entities in df['entities_trf']:
    
    filtered_entities = []
    
    for ent_text, ent_label in entities:
        
        # normalizzazione lowercase per confronto
        if ent_text.lower() not in exclude_keywords:
            all_labels.append(ent_label)
            all_entities.append((ent_text, ent_label))
            filtered_entities.append((ent_text, ent_label))
    
    filtered_entities_column.append(filtered_entities)

# Sostituiamo temporaneamente le entità filtrate
df['entities_trf_filtered'] = filtered_entities_column

label_counts = Counter(all_labels)

label_df = pd.DataFrame(label_counts.items(), columns=['Label', 'Count'])
label_df = label_df.sort_values(by='Count', ascending=False)

top12_df = label_df.head(12)

df['n_entities'] = df['entities_trf_filtered'].apply(len)

media_entita = df['n_entities'].mean()
media_display = round(media_entita, 2)

total_entities = len(all_labels)
org_entities = label_counts.get("ORG", 0)
other_entities = total_entities - org_entities

perc_org = round((org_entities / total_entities) * 100, 2)
perc_other = round((other_entities / total_entities) * 100, 2)

org_list = [
    ent_text for ent_text, ent_label in all_entities
    if ent_label == "ORG"
]

org_counts = Counter(org_list)

top10_org = pd.DataFrame(
    org_counts.most_common(10),
    columns=['Organizzazione', 'Frequenza']
)

fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1
axs[0, 0].bar(top12_df['Label'], top12_df['Count'])
axs[0, 0].set_title("Top 12 etichette NER (filtro Dell)")
axs[0, 0].tick_params(axis='x', rotation=45)

# Plot 2
axs[0, 1].hist(df['n_entities'], bins=20)
axs[0, 1].axvline(media_entita, linestyle='dashed')
axs[0, 1].set_title("Numero di entità per tweet (filtro Dell)")
axs[0, 1].text(
    0.95, 0.95,
    f"Numero medio di entità per tweet:\n{media_display}",
    transform=axs[0, 1].transAxes,
    ha='right',
    va='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)

# Plot 3
axs[1, 0].pie(
    [org_entities, other_entities],
    labels=[f"ORG ({perc_org})", f"Altre ({perc_other})"],
    startangle=90
)
axs[1, 0].set_title("Distribuzione ORG vs Altre (filtro Dell)")

# Plot 4
axs[1, 1].bar(top10_org['Organizzazione'], top10_org['Frequenza'])
axs[1, 1].set_title("Top 10 entità ORG (filtro Dell)")
axs[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()